# Optimización de Modelos Conjunto Soleado por GMM

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
#Importaciones
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

#Lectura de datos
datos = pd.read_excel('/content/drive/MyDrive/Tesina/03_Clusterizacion.xlsx')
datos.head(24)

,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM
0,2022-09-01 00:00:00,0.000000,19,7,77,0,4,15,0,Noche,Noche
1,2022-09-01 01:00:00,0.000000,19,7,82,0,4,16,1,Noche,Noche
2,2022-09-01 02:00:00,0.000000,18,9,85,0,3,16,2,Noche,Noche
3,2022-09-01 03:00:00,0.000000,18,11,87,0,3,16,3,Noche,Noche
4,2022-09-01 04:00:00,0.000000,18,11,88,0,3,16,4,Noche,Noche
5,2022-09-01 05:00:00,0.000000,17,15,86,0,3,14,5,Noche,Noche
6,2022-09-01 06:00:00,0.000000,18,47,89,0,3,16,6,Nublado,Lluvioso
7,2022-09-01 07:00:00,6.584959,18,51,95,0,4,17,7,Nublado,Lluvioso
8,2022-09-01 08:00:00,560.422022,18,47,100,0,3,18,8,Nublado,Lluvioso
9,2022-09-01 09:00:00,7720.582326,18,5,100,1,4,18,9,Nublado,Lluvioso


In [4]:
datos["Generacion_prev_hour"] = datos["Generación"].shift(1)
datos["Generacion_prev_day"] = datos["Generación"].shift(24)
datos = datos.dropna(how="any", axis= 0)

Definimos X y y

In [5]:
datos_dia = datos[datos["Cluster GMM"] == "Soleado"].copy()
datos_dia.head(10)

,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day
37,2022-09-02 13:00:00,20596.278869,23,0,45,5,2,11,13,Soleado,Soleado,27523.885172,25478.471342
207,2022-09-09 15:00:00,26098.851182,24,0,43,5,2,11,15,Soleado,Soleado,24394.058046,18959.491592
208,2022-09-09 16:00:00,28500.000000,25,0,39,4,2,10,16,Soleado,Soleado,26098.851182,16494.208425
209,2022-09-09 17:00:00,28500.000000,25,0,38,3,2,10,17,Soleado,Soleado,28500.000000,8718.476156
210,2022-09-09 18:00:00,28500.000000,26,0,36,2,2,10,18,Soleado,Soleado,28500.000000,23968.849012
211,2022-09-09 19:00:00,17303.091873,25,0,37,1,2,9,19,Soleado,Soleado,28500.000000,17559.996819
212,2022-09-09 20:00:00,2230.131403,24,0,40,0,2,9,20,Soleado,Soleado,17303.091873,2087.543919
229,2022-09-10 13:00:00,28500.000000,22,0,53,10,2,12,13,Soleado,Soleado,28976.805233,14247.046020
230,2022-09-10 14:00:00,28500.000000,23,0,47,11,2,11,14,Soleado,Soleado,28500.000000,24394.058046
231,2022-09-10 15:00:00,27067.944242,24,0,43,10,2,11,15,Soleado,Soleado,28500.000000,26098.851182


In [6]:
columns = datos_dia.drop(columns=["Fecha", "Generación", "Cluster KMeans", "Cluster GMM"]).columns

In [7]:
X = datos_dia[columns]
X

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
37,23,0,45,5,2,11,13,27523.885172,25478.471342
207,24,0,43,5,2,11,15,24394.058046,18959.491592
208,25,0,39,4,2,10,16,26098.851182,16494.208425
209,25,0,38,3,2,10,17,28500.000000,8718.476156
210,26,0,36,2,2,10,18,28500.000000,23968.849012
...,...,...,...,...,...,...,...,...,...
18280,25,0,34,5,3,8,15,25399.000000,25443.000000
18281,26,0,31,4,2,8,16,25562.000000,25385.000000
18282,26,0,32,2,1,8,17,25386.000000,22664.000000
18283,25,0,33,1,1,8,18,22872.000000,15736.000000


In [8]:
y = datos_dia[['Generación']]
y

,Generación
37,20596.278869
207,26098.851182
208,28500.000000
209,28500.000000
210,28500.000000
...,...
18280,25562.000000
18281,25386.000000
18282,22872.000000
18283,15825.000000


Dividimos entrenamiento, validación y prueba

In [9]:
train_size = int(0.7 * len(X))
val_size = int(0.85 * len(X))

In [10]:
# Entrenamiento, validación y prueba, 75, 15 y 15
X_train, y_train =  X.iloc[:train_size, :], y.iloc[:train_size, :]
X_val, y_val = X.iloc[train_size:val_size, :], y.iloc[train_size:val_size, :]
X_test, y_test = X.iloc[val_size:, :],  y.iloc[val_size:,:]

print(f'X_train: {len(X_train)}, y_train: {len(y_train)}')
print(f'X_val: {len(X_val)}, y_val: {len(y_val)}')
print(f'X_test: {len(X_test)}, y_test: {len(y_test)}')

X_train: 2412, y_train: 2412
X_val: 517, y_val: 517
X_test: 518, y_test: 518


## Escalar con MinMaxScaler

In [11]:
from sklearn.preprocessing import MinMaxScaler

In [12]:
x_scaler = MinMaxScaler().fit(X_train)
x_scaler

MinMaxScaler()

In [13]:
X_train_scaled = x_scaler.transform(X_train)
print(X_train_scaled)
print(X_train_scaled.shape)

[[0.39130435 0.         0.73076923 ... 0.46666667 0.91746284 0.84928238]
 [0.43478261 0.         0.69230769 ... 0.6        0.81313527 0.63198305]
 [0.47826087 0.         0.61538462 ... 0.66666667 0.86996171 0.54980695]
 ...
 [0.30434783 0.         0.40384615 ... 0.46666667 0.76026667 0.73196667]
 [0.34782609 0.         0.40384615 ... 0.53333333 0.73196667 0.70156667]
 [0.39130435 0.         0.42307692 ... 0.6        0.70156667 0.7206    ]]
(2412, 9)


In [14]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_train_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
37,0.391304,0.0,0.730769,0.357143,0.333333,0.6875,0.466667,0.917463,0.849282
207,0.434783,0.0,0.692308,0.357143,0.333333,0.6875,0.600000,0.813135,0.631983
208,0.478261,0.0,0.615385,0.285714,0.333333,0.6250,0.666667,0.869962,0.549807
209,0.478261,0.0,0.596154,0.214286,0.333333,0.6250,0.733333,0.950000,0.290616
210,0.521739,0.0,0.557692,0.142857,0.333333,0.6250,0.800000,0.950000,0.798962
...,...,...,...,...,...,...,...,...,...
12286,0.260870,0.0,0.423077,0.000000,0.000000,0.1250,1.000000,0.000000,0.000000
12301,0.217391,0.0,0.442308,0.357143,0.000000,0.0625,0.400000,0.783667,0.760267
12302,0.304348,0.0,0.403846,0.357143,0.000000,0.1250,0.466667,0.760267,0.731967
12303,0.347826,0.0,0.403846,0.357143,0.000000,0.1875,0.533333,0.731967,0.701567


In [15]:
X_val_scaled = x_scaler.transform(X_val)
print(X_val_scaled)
print(X_val_scaled.shape)

[[0.47826087 0.         0.44230769 ... 0.66666667 0.7034     0.7193    ]
 [0.39130435 0.         0.57692308 ... 0.73333333 0.65726667 0.65726667]
 [0.34782609 0.         0.71153846 ... 0.8        0.6976     0.31023333]
 ...
 [0.82608696 0.         0.09615385 ... 0.46666667 0.77843333 0.77773333]
 [0.34782609 0.         0.34615385 ... 0.2        0.48006667 0.69496667]
 [0.56521739 0.         0.21153846 ... 0.26666667 0.81946667 0.7502    ]]
(517, 9)


In [16]:
X_val_scaled_df = pd.DataFrame(X_val_scaled, index=X_val.index, columns=X_val.columns)
X_val_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
12305,0.478261,0.0,0.442308,0.214286,0.000000,0.3750,0.666667,0.703400,0.719300
12306,0.391304,0.0,0.576923,0.142857,0.000000,0.5000,0.733333,0.657267,0.657267
12307,0.347826,0.0,0.711538,0.071429,0.000000,0.5625,0.800000,0.697600,0.310233
12423,0.217391,0.0,0.788462,0.357143,0.333333,0.5000,0.533333,0.792533,0.490433
12424,0.304348,0.0,0.634615,0.285714,0.333333,0.4375,0.600000,0.783967,0.504200
...,...,...,...,...,...,...,...,...,...
14940,0.608696,0.0,0.250000,0.642857,0.000000,0.2500,0.333333,0.750200,0.785633
14941,0.739130,0.0,0.153846,0.857143,0.000000,0.1250,0.400000,0.766967,0.786967
14942,0.826087,0.0,0.096154,1.000000,0.000000,0.0000,0.466667,0.778433,0.777733
14962,0.347826,0.0,0.346154,0.214286,0.000000,0.0625,0.200000,0.480067,0.694967


In [17]:
X_test_scaled = x_scaler.transform(X_test)
print(X_test_scaled)
print(X_test_scaled.shape)

[[0.69565217 0.         0.11538462 ... 0.33333333 0.92343333 0.76696667]
 [0.82608696 0.         0.03846154 ... 0.4        0.95496667 0.77843333]
 [0.91304348 0.         0.         ... 0.46666667 0.95866667 0.78513333]
 ...
 [0.52173913 0.         0.48076923 ... 0.73333333 0.8462     0.75546667]
 [0.47826087 0.         0.5        ... 0.8        0.7624     0.52453333]
 [0.39130435 0.         0.59615385 ... 0.86666667 0.5275     0.0469    ]]
(518, 9)


In [18]:
X_test_scaled_df = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
X_test_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
14964,0.695652,0.0,0.115385,0.642857,0.000000,0.0625,0.333333,0.923433,0.766967
14965,0.826087,0.0,0.038462,0.857143,0.000000,0.2500,0.400000,0.954967,0.778433
14966,0.913043,0.0,0.000000,1.000000,0.000000,0.3750,0.466667,0.958667,0.785133
14967,0.956522,0.0,-0.019231,0.857143,0.000000,0.3750,0.533333,0.960967,0.775267
14968,1.000000,0.0,-0.019231,0.642857,0.000000,0.3750,0.600000,0.860433,0.873567
...,...,...,...,...,...,...,...,...,...
18280,0.478261,0.0,0.519231,0.357143,0.666667,0.5000,0.600000,0.846633,0.848100
18281,0.521739,0.0,0.461538,0.285714,0.333333,0.5000,0.666667,0.852067,0.846167
18282,0.521739,0.0,0.480769,0.142857,0.000000,0.5000,0.733333,0.846200,0.755467
18283,0.478261,0.0,0.500000,0.071429,0.000000,0.5000,0.800000,0.762400,0.524533


In [19]:
x_scaller_all = MinMaxScaler().fit(X)
print(x_scaller_all)

MinMaxScaler()


In [20]:
X_scaled = x_scaller_all.transform(X)
print(X_scaled)
print(X_scaled.shape)

[[0.36       0.         0.73584906 ... 0.46666667 0.91746284 0.84928238]
 [0.4        0.         0.69811321 ... 0.6        0.81313527 0.63198305]
 [0.44       0.         0.62264151 ... 0.66666667 0.86996171 0.54980695]
 ...
 [0.48       0.         0.49056604 ... 0.73333333 0.8462     0.75546667]
 [0.44       0.         0.50943396 ... 0.8        0.7624     0.52453333]
 [0.36       0.         0.60377358 ... 0.86666667 0.5275     0.0469    ]]
(3447, 9)


In [21]:
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)
X_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
37,0.36,0.0,0.735849,0.357143,0.333333,0.6875,0.466667,0.917463,0.849282
207,0.40,0.0,0.698113,0.357143,0.333333,0.6875,0.600000,0.813135,0.631983
208,0.44,0.0,0.622642,0.285714,0.333333,0.6250,0.666667,0.869962,0.549807
209,0.44,0.0,0.603774,0.214286,0.333333,0.6250,0.733333,0.950000,0.290616
210,0.48,0.0,0.566038,0.142857,0.333333,0.6250,0.800000,0.950000,0.798962
...,...,...,...,...,...,...,...,...,...
18280,0.44,0.0,0.528302,0.357143,0.666667,0.5000,0.600000,0.846633,0.848100
18281,0.48,0.0,0.471698,0.285714,0.333333,0.5000,0.666667,0.852067,0.846167
18282,0.48,0.0,0.490566,0.142857,0.000000,0.5000,0.733333,0.846200,0.755467
18283,0.44,0.0,0.509434,0.071429,0.000000,0.5000,0.800000,0.762400,0.524533


In [22]:
y_scaler = MinMaxScaler().fit(y_train)
print(y_scaler)

MinMaxScaler()


In [23]:
y_train_scaled = y_scaler.transform(y_train)
print(y_train_scaled)
print(y_train_scaled.shape)

[[0.68654263]
 [0.86996171]
 [0.95      ]
 ...
 [0.73196667]
 [0.70156667]
 [0.7034    ]]
(2412, 1)


In [24]:
y_train_scaled_df = pd.DataFrame(y_train_scaled, index=y_train.index, columns=y_train.columns)
y_train_scaled_df

,Generación
37,0.686543
207,0.869962
208,0.950000
209,0.950000
210,0.950000
...,...
12286,0.000000
12301,0.760267
12302,0.731967
12303,0.701567


In [25]:
y_val_scaled = y_scaler.transform(y_val)
print(y_val_scaled)
print(y_val_scaled.shape)

[[6.57266667e-01]
 [6.97600000e-01]
 [2.98400000e-01]
 [7.83966667e-01]
 [7.82933333e-01]
 [7.82600000e-01]
 [7.53100000e-01]
 [4.56700000e-01]
 [4.34666667e-02]
 [0.00000000e+00]
 [1.00000000e+00]
 [1.00000000e+00]
 [1.00000000e+00]
 [1.00000000e+00]
 [1.00000000e+00]
 [1.00000000e+00]
 [1.00000000e+00]
 [2.00766667e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [1.00000000e+00]
 [2.29400000e-01]
 [1.00000000e+00]
 [1.00000000e+00]
 [1.00000000e+00]
 [1.00000000e+00]
 [1.00000000e+00]
 [6.56666667e-01]
 [7.13833333e-01]
 [6.27166667e-01]
 [6.26333333e-01]
 [6.26066667e-01]
 [5.27166667e-01]
 [3.19666667e-01]
 [2.64000000e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [6.57766667e-01]
 [6.34033333e-01]
 [6.36166667e-01]
 [6.55400000e-01]
 [7.37566667e-01]
 [6.63633333e-01]
 [6.37766667e-01]
 [8.02466667e-01]
 [7.83966667e-01]
 [8.05566667e-01]
 [8.13566667e-01]
 [7.97433333e-01]
 [4.59833333e-01]
 [4.69333333e-02]
 [0.00000000e+00]
 [8.12700000e-01]
 [7.91266667e-01]
 [7.82000000e-01]
 [7.752000

In [26]:
y_val_scaled_df = pd.DataFrame(y_val_scaled, index=y_val.index, columns=y_val.columns)
y_val_scaled_df

,Generación
12305,0.657267
12306,0.697600
12307,0.298400
12423,0.783967
12424,0.782933
...,...
14940,0.766967
14941,0.778433
14942,0.785133
14962,0.819467


In [27]:
y_test_scaled = y_scaler.transform(y_test)
print(y_test_scaled)
print(y_test_scaled.shape)

[[0.95496667]
 [0.95866667]
 [0.96096667]
 [0.86043333]
 [0.85716667]
 [0.74293333]
 [0.73216667]
 [0.59493333]
 [0.21896667]
 [0.00806667]
 [0.        ]
 [0.93476667]
 [0.942     ]
 [0.92893333]
 [0.9326    ]
 [0.9653    ]
 [0.95383333]
 [0.85996667]
 [0.72946667]
 [0.2737    ]
 [0.9442    ]
 [0.93683333]
 [0.92893333]
 [0.9326    ]
 [0.97063333]
 [0.95473333]
 [0.8543    ]
 [0.71996667]
 [0.2737    ]
 [0.0095    ]
 [0.        ]
 [0.94696667]
 [0.9369    ]
 [0.92893333]
 [0.9326    ]
 [0.97063333]
 [0.95686667]
 [0.72766667]
 [0.01016667]
 [0.        ]
 [0.72796667]
 [0.751     ]
 [0.9429    ]
 [0.93506667]
 [0.885     ]
 [0.32096667]
 [0.0174    ]
 [0.        ]
 [0.95563333]
 [0.98136667]
 [0.97883333]
 [0.94683333]
 [0.92646667]
 [0.2737    ]
 [0.02876667]
 [0.        ]
 [0.9025    ]
 [0.9426    ]
 [0.93683333]
 [0.93753333]
 [0.9363    ]
 [0.92646667]
 [0.01706667]
 [0.        ]
 [0.90996667]
 [0.9332    ]
 [0.94056667]
 [0.93186667]
 [0.95676667]
 [0.83383333]
 [0.01043333]
 [0.  

In [28]:
y_test_scaled_df = pd.DataFrame(y_test_scaled, index=y_test.index, columns=y_test.columns)
y_test_scaled_df

,Generación
14964,0.954967
14965,0.958667
14966,0.960967
14967,0.860433
14968,0.857167
...,...
18280,0.852067
18281,0.846200
18282,0.762400
18283,0.527500


In [29]:
y_scaller_all = MinMaxScaler().fit(y)
print(y_scaller_all)

MinMaxScaler()


In [30]:
y_scaled = y_scaller_all.transform(y)
print(y_scaled)
print(y_scaled.shape)

[[0.68654263]
 [0.86996171]
 [0.95      ]
 ...
 [0.7624    ]
 [0.5275    ]
 [0.04833333]]
(3447, 1)


In [31]:
y_scaled_df = pd.DataFrame(y_scaled, index=y.index, columns=y.columns)
y_scaled_df

,Generación
37,0.686543
207,0.869962
208,0.950000
209,0.950000
210,0.950000
...,...
18280,0.852067
18281,0.846200
18282,0.762400
18283,0.527500


## Preparación para Redes Neuronales

In [32]:
import numpy as np
import pandas as pd

def create_sliding_window_with_index(data_X, data_y, lookback):
    X, y, indices = [], [], []

    # Asegurar que `data_y` tiene los mismos índices que `data_X`
    data_y = data_y.reindex(data_X.index)

    max_index = len(data_X) - lookback

    for i in range(max_index):
        X.append(data_X.iloc[i:i + lookback].values)  # Ventana de entrada

        # Obtener el índice correcto en `data_y`
        y_index = data_X.index[i + lookback]

        # Extraer el valor correspondiente de `data_y`
        if y_index in data_y.index:
            y_value = data_y.loc[y_index]
            if isinstance(y_value, pd.Series):  # Si devuelve una serie, extraer el valor
                y_value = y_value.iloc[0]
        else:
            y_value = np.nan  # Si no está, asignamos NaN

        y.append(y_value)
        indices.append(y_index)  # 🔹 Guardamos el índice original de `data_y`

    # Convertimos `X` en un array y `y` en DataFrame conservando sus índices originales
    X_array = np.array(X)
    y_df = pd.DataFrame(y, index=indices, columns=['y'])  # 🔹 Conservamos los índices originales

    return X_array, y_df


In [33]:
lookback = 48  # Puedes ajustar a 24, 72, etc.

# Aplicar la ventana deslizante a cada conjunto
X_train_windowed, y_train_windowed = create_sliding_window_with_index(X_train_scaled_df, y_train_scaled_df, lookback)
X_val_windowed, y_val_windowed = create_sliding_window_with_index(X_val_scaled_df, y_val_scaled_df, lookback)
X_test_windowed, y_test_windowed = create_sliding_window_with_index(X_test_scaled_df, y_test_scaled_df, lookback)


In [34]:
print(f'X_train: {X_train_windowed.shape}, y_train: {y_train_windowed.shape}')
print(f'X_val: {X_val_windowed.shape}, y_val: {y_val_windowed.shape}')
print(f'X_test: {X_test_windowed.shape}, y_test: {y_test_windowed.shape}')

X_train: (2364, 48, 9), y_train: (2364, 1)
X_val: (469, 48, 9), y_val: (469, 1)
X_test: (470, 48, 9), y_test: (470, 1)


## Optuna

In [36]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.8/231.8 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 10.1 MB/s eta 0:00:00


In [37]:
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
import optuna
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import seaborn as sns
from sklearn.metrics import mean_absolute_percentage_error as mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error as mean_absolute_error
from sklearn.metrics import mean_squared_error as mean_squared_error
from sklearn.metrics import r2_score as r2_score
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

### Optimización de LightGBM

In [38]:
import optuna
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error

# Definir la función de optimización
def objective(trial):
    # Espacio de búsqueda para los hiperparámetros
    num_leaves = trial.suggest_int('num_leaves', 10, 1000)
    subsample = trial.suggest_float('subsample', 0.1, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.1, 1.0)
    min_data_in_leaf = trial.suggest_int('min_data_in_leaf', 10, 100)

    # Modelo LightGBM con los hiperparámetros sugeridos
    model = LGBMRegressor(
        num_leaves=num_leaves,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_data_in_leaf=min_data_in_leaf,
        random_state=42
    )

    # Entrenamos el modelo con los datos de entrenamiento
    model.fit(X_train_scaled_df, y_train_scaled_df)

    # Predicciones en el conjunto de validación
    y_pred = model.predict(X_val_scaled_df)
    rmse = mean_squared_error(y_val_scaled_df, y_pred)

    return rmse  # Optuna minimizará este valor

# Ejecutar la optimización con 50 iteraciones
LightGBM_study = optuna.create_study(direction="minimize")
LightGBM_study.optimize(objective, n_trials=50)

# Obtener los mejores hiperparámetros
best_params = LightGBM_study.best_params
print("Mejores hiperparámetros:", best_params)

[I 2025-03-13 17:41:05,912] A new study created in memory with name: no-name-ae5bd9da-08bc-42a9-aaea-3c7d26a51243
[I 2025-03-13 17:41:05,999] Trial 0 finished with value: 0.005053830810210014 and parameters: {'num_leaves': 40, 'subsample': 0.46769806171947126, 'colsample_bytree': 0.38907392041905153, 'min_data_in_leaf': 94}. Best is trial 0 with value: 0.005053830810210014.
[I 2025-03-13 17:41:06,065] Trial 1 finished with value: 0.0053627499270276985 and parameters: {'num_leaves': 43, 'subsample': 0.26552584667540846, 'colsample_bytree': 0.6257120878901277, 'min_data_in_leaf': 27}. Best is trial 0 with value: 0.005053830810210014.
[I 2025-03-13 17:41:06,116] Trial 2 finished with value: 0.0058207950192409146 and parameters: {'num_leaves': 162, 'subsample': 0.4089570208644271, 'colsample_bytree': 0.5221221554725709, 'min_data_in_leaf': 59}. Best is trial 0 with value: 0.005053830810210014.


[LightGBM] [Warning] min_data_in_leaf is set=94, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=94
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=94, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=94
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000447 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 634
[LightGBM] [Info] Number of data points in the train set: 2412, number of used features: 8
[LightGBM] [Info] Start training from score 0.648351
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posi

[I 2025-03-13 17:41:06,318] Trial 3 finished with value: 0.005358961389584326 and parameters: {'num_leaves': 572, 'subsample': 0.256627196821234, 'colsample_bytree': 0.7817576714589946, 'min_data_in_leaf': 13}. Best is trial 0 with value: 0.005053830810210014.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-03-13 17:41:06,358] Trial 4 finished with value: 0.005824259023579439 and parameters: {'num_leaves': 830, 'subsample': 0.44000437297501327, 'colsample_bytree': 0.13853902945447685, 'min_data_in_leaf': 90}. Best is trial 0 with value: 0.005053830810210014.
[I 2025-03-13 17:41:06,387] Trial 5 finished with value: 0.005715978872214999 and parameters: {'num_leaves': 292, 'subsample': 0.2251642163354655, 'colsample_bytree': 0.18041243084252656, 'min_data_in_leaf': 96}. Best is trial 0 with value: 0.005053830810210014.
[I 2025-03-13 17:41:06,444] Trial 6 finished with value: 0.0051358263898920314 and parameters: {'num_leaves': 426, 'subsample': 0.22943814362439355, 'colsample_bytree': 0.5778450983935197, 'min_data_in_leaf': 57}. Best is trial 0 with value: 0.005053830810210014.
[I 2025-03-13 17:41:06,514] Trial 7 finished with value: 0.004668756599975616 and parameters: {'num_leaves': 290, 'subsample': 0.25742627337666507, 'colsample_bytree': 0.8428767194733862, 'min_data_in_leaf': 4

...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-03-13 17:41:09,429] Trial 45 finished with value: 0.004678150876633406 and parameters: {'num_leaves': 993, 'subsample': 0.6690845751940795, 'colsample_bytree': 0.8151947063002407, 'min_data_in_leaf': 86}. Best is trial 17 with value: 0.004426787839221232.
[I 2025-03-13 17:41:09,510] Trial 46 finished with value: 0.004685935193378671 and parameters: {'num_leaves': 883, 'subsample': 0.8708057751981974, 'colsample_bytree': 0.9582080434722052, 'min_data_in_leaf': 71}. Best is trial 17 with value: 0.004426787839221232.
[I 2025-03-13 17:41:09,574] Trial 47 finished with value: 0.004988527794554446 and parameters: {'num_leaves': 838, 'subsample': 0.36521217962777475, 'colsample_bytree': 0.6779830576730516, 'min_data_in_leaf': 100}. Best is trial 17 with value: 0.004426787839221232.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-03-13 17:41:09,649] Trial 48 finished with value: 0.004536360968780447 and parameters: {'num_leaves': 938, 'subsample': 0.7965727851330713, 'colsample_bytree': 0.7350966245491608, 'min_data_in_leaf': 78}. Best is trial 17 with value: 0.004426787839221232.
[I 2025-03-13 17:41:09,797] Trial 49 finished with value: 0.006760130757792088 and parameters: {'num_leaves': 660, 'subsample': 0.7750809868359111, 'colsample_bytree': 0.5578169301706375, 'min_data_in_leaf': 19}. Best is trial 17 with value: 0.004426787839221232.


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=78, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=78
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000142 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 634
[LightGBM] [Info] Number of data points in the train set: 2412, number of used features: 8
[LightGBM] [Info] Start training from score 0.648351
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM

### Random Forest

In [39]:
def objective(trial):
    # Definir los hiperparámetros a optimizar
    n_estimators = trial.suggest_int("n_estimators", 100, 500, step=50)
    max_depth = trial.suggest_int("max_depth", 10, 50, step=5)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    bootstrap = trial.suggest_categorical("bootstrap", [True, False])

    # Definir el modelo con los hiperparámetros actuales
    RF_model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        bootstrap=bootstrap,
        criterion="squared_error",
        random_state=0
    )

    # Entrenar el modelo
    RF_model.fit(X_train_scaled_df, y_train_scaled_df)

    # Predicciones en el conjunto de validación
    y_pred = RF_model.predict(X_val_scaled_df)
    rmse = mean_squared_error(y_val_scaled_df, y_pred)

    return rmse  # Optuna minimizará este valor

# Crear el estudio de optimización
RF_study = optuna.create_study(direction="minimize")
RF_study.optimize(objective, n_trials=50)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", RF_study.best_params)

[I 2025-03-13 17:41:18,388] A new study created in memory with name: no-name-336b0d7f-76d9-40f1-a556-f8e0a7606910
[I 2025-03-13 17:41:20,126] Trial 0 finished with value: 0.0068558634572856245 and parameters: {'n_estimators': 150, 'max_depth': 25, 'min_samples_split': 19, 'min_samples_leaf': 1, 'bootstrap': False}. Best is trial 0 with value: 0.0068558634572856245.
[I 2025-03-13 17:41:21,172] Trial 1 finished with value: 0.004117203687068326 and parameters: {'n_estimators': 150, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 8, 'bootstrap': True}. Best is trial 1 with value: 0.004117203687068326.
[I 2025-03-13 17:41:22,749] Trial 2 finished with value: 0.0056059406337417616 and parameters: {'n_estimators': 150, 'max_depth': 45, 'min_samples_split': 8, 'min_samples_leaf': 6, 'bootstrap': False}. Best is trial 1 with value: 0.004117203687068326.
[I 2025-03-13 17:41:24,231] Trial 3 finished with value: 0.005430745796289882 and parameters: {'n_estimators': 150, 'max_depth': 2

Mejores hiperparámetros: {'n_estimators': 350, 'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 10, 'bootstrap': True}


### CTNET

In [40]:
def compile_and_fit(model, xtrain=X_train_windowed, ytrain=y_train_windowed, learning_rate=0.0001):
    model.compile(loss=[tf.keras.losses.MeanSquaredError()],
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=[tf.keras.metrics.RootMeanSquaredError(), tf.keras.metrics.MeanAbsolutePercentageError(), tf.keras.metrics.MeanAbsoluteError()])

    history = model.fit(xtrain, ytrain, epochs=50,
                        batch_size=512, validation_split=0.2, verbose=1)
    return history

def Loss(train_loss, valid_loss):
    plt.plot(train_loss)
    plt.plot(valid_loss)
    plt.rcParams["figure.figsize"] = (15, 3)
    plt.title('Model Losses')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train Loss', 'Validation Loss'], loc='upper left')
    plt.savefig('out/loss_plot.png')
    plt.show()

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = layers.LayerNormalization()(inputs)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=128, kernel_size=2, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(norm_x, norm_x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    return norm_x

# Construir modelo con hiperparámetros sugeridos
def build_model(input_shape, head_size, num_heads, ff_dim, num_transformer_blocks, mlp_units, dropout, mlp_dropout):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs

    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)

    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.GlobalAveragePooling1D(data_format="channels_first")(x)

    for units in mlp_units:
        x = layers.Dense(units, activation="relu")(x)

    x = layers.Dropout(mlp_dropout)(x)
    outputs = layers.Dense(1)(x)
    return tf.keras.Model(inputs, outputs)

In [41]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [42]:
# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    head_size = trial.suggest_int("head_size", 2, 8)
    num_heads = trial.suggest_int("num_heads", 2, 8)
    ff_dim = trial.suggest_int("ff_dim", 16, 128, step=16)
    num_transformer_blocks = trial.suggest_int("num_transformer_blocks", 1, 5)
    mlp_units = [trial.suggest_int("mlp_units_1", 64, 512, step=64),
                 trial.suggest_int("mlp_units_2", 32, 256, step=32)]
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    mlp_dropout = trial.suggest_float("mlp_dropout", 0.1, 0.5)
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    # Construir el modelo
    model = build_model(
        input_shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]),
        head_size=head_size,
        num_heads=num_heads,
        ff_dim=ff_dim,
        num_transformer_blocks=num_transformer_blocks,
        mlp_units=mlp_units,
        dropout=dropout,
        mlp_dropout=mlp_dropout
    )

    # Compilar y entrenar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se puede detener antes con Early Stopping
        batch_size=batch_size,  # 🔹 Batch size optimizado
        validation_data=(X_val_windowed, y_val_windowed),  # 🔹 Validación en cada época
        callbacks=[early_stop],  # 🔹 Early Stopping activado
        verbose=0
    )

    # Obtener pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Minimizar MSE

# Crear el estudio de optimización
CTNET_study = optuna.create_study(direction="minimize")
CTNET_study.optimize(objective, n_trials=50)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", CTNET_study.best_params)


[I 2025-03-13 17:43:37,512] A new study created in memory with name: no-name-faf02d41-58ab-42ec-b552-62d0e7d44c9e
[I 2025-03-13 17:44:58,742] Trial 0 finished with value: 0.07055661827325821 and parameters: {'head_size': 4, 'num_heads': 7, 'ff_dim': 32, 'num_transformer_blocks': 5, 'mlp_units_1': 192, 'mlp_units_2': 256, 'dropout': 0.31297654581517925, 'mlp_dropout': 0.11178200242599656, 'learning_rate': 0.0041039590735889725, 'batch_size': 128}. Best is trial 0 with value: 0.07055661827325821.
[I 2025-03-13 17:45:50,871] Trial 1 finished with value: 0.0634951964020729 and parameters: {'head_size': 3, 'num_heads': 2, 'ff_dim': 96, 'num_transformer_blocks': 2, 'mlp_units_1': 512, 'mlp_units_2': 192, 'dropout': 0.31513820768663703, 'mlp_dropout': 0.45451740563951404, 'learning_rate': 0.0001768532472600309, 'batch_size': 128}. Best is trial 1 with value: 0.0634951964020729.
[I 2025-03-13 17:47:00,947] Trial 2 finished with value: 0.10293217748403549 and parameters: {'head_size': 4, 'num_h

Mejores hiperparámetros: {'head_size': 2, 'num_heads': 5, 'ff_dim': 48, 'num_transformer_blocks': 2, 'mlp_units_1': 448, 'mlp_units_2': 32, 'dropout': 0.3565691296343225, 'mlp_dropout': 0.43344984355636373, 'learning_rate': 0.001443152051243645, 'batch_size': 512}


### Forescasting

In [43]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, Conv1D, BatchNormalization, MaxPooling1D, Bidirectional, LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam

# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    filters = trial.suggest_categorical("filters", [32, 64, 128])
    kernel_size = trial.suggest_int("kernel_size", 2, 5)
    lstm_units_1 = trial.suggest_categorical("lstm_units_1", [64, 128, 256])
    lstm_units_2 = trial.suggest_categorical("lstm_units_2", [32, 64, 128])
    lstm_units_3 = trial.suggest_categorical("lstm_units_3", [16, 32, 64])
    dropout_lstm = trial.suggest_float("dropout_lstm", 0.1, 0.5)
    dropout_dense = trial.suggest_float("dropout_dense", 0.1, 0.5)
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])  # 🔹 Optimización del batch size

    # Construcción del modelo
    model = Sequential()
    model.add(InputLayer((X_train_windowed.shape[1], X_train_windowed.shape[2])))

    # CNN
    model.add(Conv1D(filters=filters, kernel_size=kernel_size, padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))

    # BiLSTM
    model.add(Bidirectional(LSTM(lstm_units_1, return_sequences=True)))
    model.add(Bidirectional(LSTM(lstm_units_2, return_sequences=True)))
    model.add(Dropout(dropout_lstm))
    model.add(Bidirectional(LSTM(lstm_units_3, return_sequences=False)))

    # Normalización y Dropout
    model.add(BatchNormalization())
    model.add(Dropout(dropout_dense))

    # Capas Densas
    model.add(Dense(16, activation='relu'))
    model.add(Dense(1, activation='relu'))

    # Compilar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    # Entrenar el modelo con validación
    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se puede detener antes con Early Stopping
        batch_size=batch_size,  # 🔹 Batch size optimizado
        validation_data=(X_val_windowed, y_val_windowed),  # 🔹 Validación en cada época
        callbacks=[early_stop],  # 🔹 Early Stopping activado
        verbose=0
    )

    # Obtener la mejor pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Optuna minimizará este valor

# Crear el estudio de optimización
Forecasting_study = optuna.create_study(direction="minimize")
Forecasting_study.optimize(objective, n_trials=50, n_jobs=-1)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", Forecasting_study.best_params)

[I 2025-03-13 18:27:33,007] A new study created in memory with name: no-name-0f2e5c8e-69e2-4106-8523-1439817d9077
[I 2025-03-13 18:28:55,045] Trial 7 finished with value: 0.2357964813709259 and parameters: {'filters': 64, 'kernel_size': 3, 'lstm_units_1': 256, 'lstm_units_2': 64, 'lstm_units_3': 32, 'dropout_lstm': 0.4213126301129697, 'dropout_dense': 0.3581476659584125, 'learning_rate': 0.0013396132273083008, 'batch_size': 512}. Best is trial 7 with value: 0.2357964813709259.
[I 2025-03-13 18:29:07,071] Trial 12 finished with value: 0.22473730146884918 and parameters: {'filters': 128, 'kernel_size': 3, 'lstm_units_1': 128, 'lstm_units_2': 32, 'lstm_units_3': 64, 'dropout_lstm': 0.22813116026319033, 'dropout_dense': 0.45243387009341407, 'learning_rate': 0.0021975869593535043, 'batch_size': 256}. Best is trial 12 with value: 0.22473730146884918.
[I 2025-03-13 18:29:18,959] Trial 13 finished with value: 0.5417514443397522 and parameters: {'filters': 64, 'kernel_size': 2, 'lstm_units_1': 

Mejores hiperparámetros: {'filters': 64, 'kernel_size': 2, 'lstm_units_1': 64, 'lstm_units_2': 32, 'lstm_units_3': 16, 'dropout_lstm': 0.43199403176870343, 'dropout_dense': 0.1588272373910951, 'learning_rate': 0.007973597957122082, 'batch_size': 128}


### Photovoltaic

In [44]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Bidirectional, GRU, MultiHeadAttention
from tensorflow.keras.layers import Flatten, Dropout, Dense
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error

# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    filters_1 = trial.suggest_categorical("filters_1", [32, 64, 128])
    filters_2 = trial.suggest_categorical("filters_2", [64, 128, 256])
    kernel_size = trial.suggest_int("kernel_size", 2, 5)
    gru_units = trial.suggest_categorical("gru_units", [32, 64, 128])
    num_heads = trial.suggest_categorical("num_heads", [2, 4, 8])
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)
    dense_units_1 = trial.suggest_categorical("dense_units_1", [32, 64, 128])
    dense_units_2 = trial.suggest_categorical("dense_units_2", [16, 32, 64])
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    # Definir el modelo
    inputs = Input(shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]))

    # Primera capa CNN
    x = Conv1D(filters=filters_1, kernel_size=kernel_size, padding='same', activation='relu')(inputs)
    x = MaxPooling1D(pool_size=2)(x)

    # Segunda capa CNN
    x = Conv1D(filters=filters_2, kernel_size=kernel_size, padding='same', activation='relu')(x)
    x = MaxPooling1D(pool_size=2)(x)

    # Capa BiGRU
    x = Bidirectional(GRU(gru_units, return_sequences=True))(x)

    # Atención MultiHead
    attention = MultiHeadAttention(num_heads=num_heads, key_dim=128)(x, x)

    # Aplanar y Dropout
    x = Flatten()(attention)
    x = Dropout(dropout_rate)(x)
    x = Dense(dense_units_1, activation="relu", kernel_regularizer=l2(0.01))(x)
    x = Dense(dense_units_2, activation="relu")(x)

    # Capa de salida
    outputs = Dense(1, activation="linear")(x)

    # Construcción del modelo
    model = Model(inputs=inputs, outputs=outputs)

    # Compilar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    # Early Stopping
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

    # Entrenar el modelo
    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se detendrá con Early Stopping
        batch_size=batch_size,
        validation_data=(X_val_windowed, y_val_windowed),
        callbacks=[early_stop],
        verbose=0
    )

    # Obtener la mejor pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Optuna minimizará este valor

# Crear el estudio de optimización
Photovoltaic_study = optuna.create_study(direction="minimize")
Photovoltaic_study.optimize(objective, n_trials=50, n_jobs=-1)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", Photovoltaic_study.best_params)

[I 2025-03-13 18:42:16,862] A new study created in memory with name: no-name-fe4c63c4-6af0-408b-bea7-c53d4e67feb7


Epoch 11: early stopping
Restoring model weights from the end of the best epoch: 1.


[I 2025-03-13 18:43:18,116] Trial 6 finished with value: 1.043923020362854 and parameters: {'filters_1': 128, 'filters_2': 256, 'kernel_size': 5, 'gru_units': 64, 'num_heads': 2, 'dropout_rate': 0.2416218068521156, 'dense_units_1': 32, 'dense_units_2': 16, 'learning_rate': 0.009259746316330787, 'batch_size': 256}. Best is trial 6 with value: 1.043923020362854.


Epoch 71: early stopping
Restoring model weights from the end of the best epoch: 61.


[I 2025-03-13 18:43:54,542] Trial 11 finished with value: 0.060482434928417206 and parameters: {'filters_1': 64, 'filters_2': 128, 'kernel_size': 5, 'gru_units': 128, 'num_heads': 4, 'dropout_rate': 0.4210450237881984, 'dense_units_1': 128, 'dense_units_2': 16, 'learning_rate': 0.0006686601426284589, 'batch_size': 512}. Best is trial 11 with value: 0.060482434928417206.


...

Epoch 58: early stopping
Restoring model weights from the end of the best epoch: 48.


[I 2025-03-13 18:49:31,655] Trial 49 finished with value: 0.062447886914014816 and parameters: {'filters_1': 64, 'filters_2': 128, 'kernel_size': 2, 'gru_units': 32, 'num_heads': 2, 'dropout_rate': 0.3247791350613646, 'dense_units_1': 32, 'dense_units_2': 16, 'learning_rate': 0.0003721820804645685, 'batch_size': 128}. Best is trial 45 with value: 0.05489955097436905.


Epoch 75: early stopping
Restoring model weights from the end of the best epoch: 65.


[I 2025-03-13 18:49:36,063] Trial 48 finished with value: 0.06069372966885567 and parameters: {'filters_1': 64, 'filters_2': 128, 'kernel_size': 2, 'gru_units': 32, 'num_heads': 2, 'dropout_rate': 0.3260268918221032, 'dense_units_1': 32, 'dense_units_2': 16, 'learning_rate': 0.0002478618801240771, 'batch_size': 128}. Best is trial 45 with value: 0.05489955097436905.


Mejores hiperparámetros: {'filters_1': 64, 'filters_2': 64, 'kernel_size': 2, 'gru_units': 32, 'num_heads': 2, 'dropout_rate': 0.32554532521832025, 'dense_units_1': 32, 'dense_units_2': 64, 'learning_rate': 0.002609513288004872, 'batch_size': 128}
